In [3]:
import os
import math
import numpy as np
import pandas as pd

# =========================================================
# PATHS / CONFIG
# =========================================================
INPUT_DIR  = r"C:\Users\loci_\Desktop\trading_webapp\DATA\all_input_files"
OUTPUT_DIR = r"C:\Users\loci_\Desktop\trading_webapp\DATA"

RX_FILE = os.path.join(INPUT_DIR, "RX1_small.csv")
AD_FILE = os.path.join(INPUT_DIR, "AD1_small.csv")

START_NAV_USD   = 10_000_000
ANNUAL_VOL      = 0.20
TRADING_DAYS    = 256
VOL_LOOKBACK    = 36
DECAY           = 2 / (VOL_LOOKBACK + 1)   # 2/37

# Fixed PDM for sizing
PDM_CONST       = 1.6

# Instrument weights for portfolio (used for portfolio metrics)
WEIGHTS = {"RX1": 0.5, "AD1": 0.5}

# EWMA family: slow = 4 × fast, with your scalers
EWMA_SETUPS = [
    {"fast": 2,  "slow":  8,  "scaler": 10.6},
    {"fast": 4,  "slow": 16,  "scaler":  7.5},
    {"fast": 8,  "slow": 32,  "scaler":  5.3},
    {"fast": 16, "slow": 64,  "scaler":  3.75},
    {"fast": 32, "slow":128,  "scaler":  2.65},
    {"fast": 64, "slow":256,  "scaler":  1.87},
]

# =========================================================
# METRICS HELPERS
# =========================================================
def _annualize_mean(mu_daily, trading_days=TRADING_DAYS):
    return mu_daily * trading_days

def _annualize_vol(sig_daily, trading_days=TRADING_DAYS):
    return sig_daily * np.sqrt(trading_days)

def daily_vol(series):
    s = pd.Series(series).replace([np.inf, -np.inf], np.nan).dropna()
    return np.nan if len(s) < 2 else s.std(ddof=0)

def _downside_std(returns):
    r = pd.Series(returns).dropna()
    r_dn = r[r < 0]
    if len(r_dn) < 2:
        return np.nan
    return r_dn.std(ddof=0)

def _max_drawdown(returns):
    r = pd.Series(returns).fillna(0.0)
    nav = (1.0 + r).cumprod()
    peak = nav.cummax()
    dd = (nav / peak) - 1.0
    return dd.min(), dd, nav

def performance_metrics(returns, trading_days=TRADING_DAYS):
    s = pd.Series(returns).replace([np.inf, -np.inf], np.nan).dropna()
    if len(s) < 3:
        return pd.Series({
            "ann_return": np.nan, "ann_vol": np.nan,
            "sharpe": np.nan, "sortino": np.nan,
            "max_dd": np.nan, "calmar": np.nan,
            "hit_rate": np.nan, "skew": np.nan, "kurt": np.nan
        })

    mu_d   = s.mean()
    sig_d  = s.std(ddof=0)
    dn_std = _downside_std(s)

    ann_ret = _annualize_mean(mu_d, trading_days)
    ann_vol = _annualize_vol(sig_d, trading_days)
    sharpe  = np.nan if sig_d == 0 else (mu_d / sig_d) * np.sqrt(trading_days)
    sortino = np.nan if (pd.isna(dn_std) or dn_std == 0) else (mu_d / dn_std) * np.sqrt(trading_days)

    mdd, _, _ = _max_drawdown(s)
    calmar = np.nan if (pd.isna(mdd) or mdd == 0) else (ann_ret / abs(mdd))

    hit_rate = (s > 0).mean()
    skew = s.skew()
    kurt = s.kurtosis()

    return pd.Series({
        "ann_return": ann_ret,
        "ann_vol": ann_vol,
        "sharpe": sharpe,
        "sortino": sortino,
        "max_dd": mdd,
        "calmar": calmar,
        "hit_rate": hit_rate,
        "skew": skew,
        "kurt": kurt
    })

# ---------- MAE / MFE (price-based, size-independent) ----------
def extract_trades_mae_mfe(df, price_col="PX_CLOSE_1D", pos_col="current_position"):
    """
    Defines a 'trade' as a contiguous run of non-zero position with constant sign.
    Entry = close at trade start; exit at position zero/flip.
    Returns a DataFrame with price-based MAE/MFE and edge ratio.
    """
    px = df[price_col].astype(float).values
    pos = df[pos_col].astype(float).values
    dates = df.index

    trades = []
    in_trade = False
    side = 0
    entry_idx = None
    entry_price = None
    hi = lo = None

    def close_trade(exit_idx):
        nonlocal trades, in_trade, side, entry_idx, entry_price, hi, lo
        if not in_trade:
            return
        if side > 0:  # long
            mae = max(0.0, entry_price - lo)
            mfe = max(0.0, hi - entry_price)
            side_txt = "long"
        else:         # short
            mae = max(0.0, hi - entry_price)
            mfe = max(0.0, entry_price - lo)
            side_txt = "short"
        trades.append({
            "entry_date": dates[entry_idx],
            "exit_date":  dates[exit_idx],
            "side": side_txt,
            "entry_price": entry_price,
            "min_price": lo,
            "max_price": hi,
            "MAE": mae,
            "MFE": mfe,
            "edge_ratio": (np.nan if mae == 0 else mfe / mae)
        })
        in_trade = False
        side = 0
        entry_idx = None
        entry_price = None
        hi = lo = None

    for i in range(len(px)):
        sgn = 0 if np.isnan(pos[i]) else (1 if pos[i] > 0 else (-1 if pos[i] < 0 else 0))

        if not in_trade:
            if sgn != 0:
                in_trade = True
                side = sgn
                entry_idx = i
                entry_price = px[i]
                hi = px[i]
                lo = px[i]
        else:
            hi = max(hi, px[i])
            lo = min(lo, px[i])
            if sgn == 0 or sgn != side:
                close_trade(i)
                if sgn != 0 and sgn != side:
                    in_trade = True
                    side = sgn
                    entry_idx = i
                    entry_price = px[i]
                    hi = px[i]
                    lo = px[i]

    if in_trade:
        close_trade(len(px) - 1)

    return pd.DataFrame(trades)

def aggregate_mae_mfe(trades_df):
    if trades_df is None or len(trades_df) == 0:
        return pd.Series({"mae_mean": np.nan, "mfe_mean": np.nan, "edge_ratio": np.nan,
                          "mae_median": np.nan, "mfe_median": np.nan})
    mae_mean = trades_df["MAE"].mean()
    mfe_mean = trades_df["MFE"].mean()
    edge = np.nan if (mae_mean is None or mae_mean == 0 or np.isnan(mae_mean)) else (mfe_mean / mae_mean)
    return pd.Series({
        "mae_mean": mae_mean,
        "mfe_mean": mfe_mean,
        "edge_ratio": edge,
        "mae_median": trades_df["MAE"].median(),
        "mfe_median": trades_df["MFE"].median()
    })

def compute_turnover(df, trading_days=256):
    """
    Turnover = average number of lots traded yearly / (2 * average absolute lots held).
    Also returns: avg_yearly_lots, avg_abs_pos (for audit).
    """
    n = len(df)
    years = n / trading_days if n else np.nan
    avg_yearly_lots = (df["trades"].abs().sum() / years) if years and years > 0 else np.nan
    avg_abs_pos = df["current_position"].abs().mean() if "current_position" in df.columns else np.nan
    turnover = (avg_yearly_lots / (2 * avg_abs_pos)) if (avg_abs_pos and avg_abs_pos != 0) else np.nan
    return turnover, avg_yearly_lots, avg_abs_pos

# =========================================================
# ONE-INSTRUMENT / ONE-EWMA BACKTEST
# =========================================================
def run_one_ewma(df_raw, instr_name, fast, slow, scaler, weight, pdm, output_dir):
    """
    df_raw columns expected:
      Date, PX_CLOSE_1D, st_dev, Exchange rate
      (optional) TICK_VALUE, TICK_SIZE, POINT_VALUE
    """
    df = df_raw.copy()
    df = df.dropna(subset=["Date"]).sort_values("Date").set_index("Date")

    price = df["PX_CLOSE_1D"].astype(float)
    stdev_price = df["st_dev"].astype(float)

    # FX: IVV should be lower than ICV → FX column is EUR per USD (>1), so EUR→USD = divide
    fx_series = df["Exchange rate"].astype(float) if "Exchange rate" in df.columns else pd.Series(1.0, index=df.index)

    # Tick / point info
    if "TICK_VALUE" in df.columns and "TICK_SIZE" in df.columns:
        tick_value = float(df["TICK_VALUE"].iloc[0])
        tick_size  = float(df["TICK_SIZE"].iloc[0])
        MULT = tick_value / tick_size
    else:
        # sensible fallback (RX/AD-like)
        tick_value = 1000.0
        tick_size  = 0.01
        MULT = tick_value / tick_size

    # 1) returns for signal & variance
    df["ret_pct_for_sig"] = price.pct_change()
    df["ret_net_for_var"] = price.diff()

    # 2) EWMA variance (squared net returns)
    variance = [np.nan] * len(df)
    ret_net_vals = df["ret_net_for_var"].values
    first_idx = np.where(~np.isnan(ret_net_vals))[0][0]
    first_sq = ret_net_vals[first_idx] ** 2
    variance[first_idx] = first_sq
    prev_var = first_sq
    for i in range(first_idx + 1, len(df)):
        rn = ret_net_vals[i]
        rn2 = 0.0 if np.isnan(rn) else rn * rn
        new_var = DECAY * rn2 + (1 - DECAY) * prev_var
        variance[i] = new_var
        prev_var = new_var
    df["variance"] = variance
    df["stdev_variance"] = np.sqrt(df["variance"])

    # 3) EWMAs for signal
    df["ewma_fast"] = price.ewm(span=fast, adjust=False).mean()
    df["ewma_slow"] = price.ewm(span=slow,  adjust=False).mean()
    df["raw_crossover"] = df["ewma_fast"] - df["ewma_slow"]
    df["vol_adj_crossover"] = df["raw_crossover"] / df["stdev_variance"]

    # 4) Forecast (with family scaler) & cap
    df["scaled_forecast"] = df["vol_adj_crossover"] * scaler
    df["capped_forecast"] = df["scaled_forecast"].clip(-20, 20)

    # Raw signal PnL proxy
    df["forecast_x_return"] = df["capped_forecast"].shift(1) * df["ret_pct_for_sig"]

    # 5) Cash vol chain (ICV/IVV)
    df["price_volatility"] = (stdev_price / price * 100).round(2)
    df["one_pct_move"]     = price * 0.01
    if "POINT_VALUE" in df.columns:
        point_value = float(df["POINT_VALUE"].iloc[0])
    else:
        point_value = MULT
    df["block_value_eur"]  = df["one_pct_move"] * point_value
    df["icv_eur"]          = df["price_volatility"] * df["block_value_eur"]
    df["ivv"]              = df["icv_eur"] / fx_series  # EUR → USD (divide), so IVV < ICV as requested

    # 6) Iterative sizing with lagged NAV
    nav_list = []
    dcv_list = []                 # daily_cash_vol_target
    target_contracts_list = []
    trades_list = []
    current_pos_list = []
    strat_ret_list = []
    vol_scaler_list = []
    subsystem_pos_list = []
    port_instr_pos_list = []

    prev_nav = START_NAV_USD
    prev_dcv = prev_nav * ANNUAL_VOL / math.sqrt(TRADING_DAYS)
    prev_target = 0

    for i, (date, row) in enumerate(df.iterrows()):
        px_today = row["PX_CLOSE_1D"]
        if i == 0:
            delta_price = 0.0
        else:
            px_yday = df["PX_CLOSE_1D"].iloc[i - 1]
            delta_price = px_today - px_yday

        # 1) risk budget = yesterday's NAV
        dcv_today = prev_dcv

        # 2) vol scaler
        ivv_today = row["ivv"]
        vol_scaler_today = np.nan if (np.isnan(ivv_today) or ivv_today == 0) else dcv_today / ivv_today

        # 3) subsystem position
        cap_f = row["capped_forecast"]
        subsystem_pos = 0.0 if (np.isnan(cap_f) or np.isnan(vol_scaler_today)) else (cap_f * vol_scaler_today) / 10.0

        # 4) portfolio instrument position BEFORE rounding
        port_instr_pos = subsystem_pos * weight * pdm

        # 5) target contracts (rounded)
        target_contracts = int(round(port_instr_pos))
        trades = target_contracts - prev_target

        # 6) carry PnL only (no exec prices yet)
        carry_pnl_eur = prev_target * delta_price * MULT
        fx_today = fx_series.loc[date]
        carry_pnl_usd = carry_pnl_eur / fx_today   # EUR → USD (divide) with FX = EUR per USD

        pnl_usd = carry_pnl_usd
        nav_today = prev_nav + pnl_usd
        strat_ret = 0.0 if prev_nav == 0 else (pnl_usd / prev_nav)

        # store
        nav_list.append(nav_today)
        dcv_list.append(dcv_today)
        target_contracts_list.append(target_contracts)
        trades_list.append(trades)
        current_pos_list.append(target_contracts)
        strat_ret_list.append(strat_ret)
        vol_scaler_list.append(vol_scaler_today)
        subsystem_pos_list.append(subsystem_pos)
        port_instr_pos_list.append(port_instr_pos)

        # roll
        prev_nav = nav_today
        prev_dcv = nav_today * ANNUAL_VOL / math.sqrt(TRADING_DAYS)
        prev_target = target_contracts

    # 7) Assign back
    df["nav_usd"]                  = nav_list
    df["daily_cash_vol_target"]    = dcv_list
    df["volatility_scaler"]        = vol_scaler_list
    df["subsystem_position"]       = subsystem_pos_list
    df["portfolio_instr_position"] = port_instr_pos_list
    df["target_contracts"]         = target_contracts_list
    df["current_position"]         = current_pos_list
    df["trades"]                   = trades_list
    df["strategy_ret"]             = strat_ret_list
    df["cumulative_pnl_usd"]       = df["nav_usd"] - START_NAV_USD
    df["pdm_used"]                 = pdm
    df["eff_multiplier"]           = weight * pdm

    # 8) Save per-instrument CSV for this EWMA setup
    out_name = f"{instr_name}_ewma{fast}_{slow}.csv"
    out_path = os.path.join(output_dir, out_name)
    df.reset_index().to_csv(out_path, index=False)

    return {
        "df": df,
        "instrument": instr_name,
        "fast": fast, "slow": slow, "scaler": scaler,
        "csv": out_path
    }

# =========================================================
# LOAD INPUT DATA
# =========================================================
rx_raw = pd.read_csv(RX_FILE)
ad_raw = pd.read_csv(AD_FILE)

# Ensure Date parsing (day-first)
rx_raw["Date"] = pd.to_datetime(rx_raw["Date"], dayfirst=True, errors="coerce")
ad_raw["Date"] = pd.to_datetime(ad_raw["Date"], dayfirst=True, errors="coerce")

rx_raw = rx_raw.dropna(subset=["Date"]).sort_values("Date")
ad_raw = ad_raw.dropna(subset=["Date"]).sort_values("Date")

# =========================================================
# RUN SWEEP
# =========================================================
summary_rows = []

for setup in EWMA_SETUPS:
    f, s, sc = setup["fast"], setup["slow"], setup["scaler"]
    print(f"\n=== EWMA {f}/{s} (scaler={sc}) ===")

    rx_run = run_one_ewma(rx_raw, "RX1", f, s, sc, WEIGHTS["RX1"], PDM_CONST, OUTPUT_DIR)
    ad_run = run_one_ewma(ad_raw, "AD1", f, s, sc, WEIGHTS["AD1"], PDM_CONST, OUTPUT_DIR)

    rx_df = rx_run["df"]
    ad_df = ad_run["df"]

    # --- Metrics (raw & executed) for instruments
    rx_metrics = performance_metrics(rx_df["forecast_x_return"]).add_prefix("raw_")
    rx_metrics = pd.concat([rx_metrics, performance_metrics(rx_df["strategy_ret"]).add_prefix("exec_")])
    ad_metrics = performance_metrics(ad_df["forecast_x_return"]).add_prefix("raw_")
    ad_metrics = pd.concat([ad_metrics, performance_metrics(ad_df["strategy_ret"]).add_prefix("exec_")])

    # --- Volatility (daily & annualised) for instruments
    rx_vol_daily_exec = daily_vol(rx_df["strategy_ret"])
    rx_vol_ann_exec   = _annualize_vol(rx_vol_daily_exec)
    rx_vol_daily_raw  = daily_vol(rx_df["forecast_x_return"])
    rx_vol_ann_raw    = _annualize_vol(rx_vol_daily_raw)

    ad_vol_daily_exec = daily_vol(ad_df["strategy_ret"])
    ad_vol_ann_exec   = _annualize_vol(ad_vol_daily_exec)
    ad_vol_daily_raw  = daily_vol(ad_df["forecast_x_return"])
    ad_vol_ann_raw    = _annualize_vol(ad_vol_daily_raw)

    # --- MAE/MFE per instrument (price-based, size-independent)
    rx_trades = extract_trades_mae_mfe(rx_df, price_col="PX_CLOSE_1D", pos_col="current_position")
    ad_trades = extract_trades_mae_mfe(ad_df, price_col="PX_CLOSE_1D", pos_col="current_position")

    rx_trades_out = os.path.join(OUTPUT_DIR, f"RX1_trades_mae_mfe_{f}_{s}.csv")
    ad_trades_out = os.path.join(OUTPUT_DIR, f"AD1_trades_mae_mfe_{f}_{s}.csv")
    rx_trades.to_csv(rx_trades_out, index=False)
    ad_trades.to_csv(ad_trades_out, index=False)

    rx_mae_mfe = aggregate_mae_mfe(rx_trades)
    ad_mae_mfe = aggregate_mae_mfe(ad_trades)

    # --- Turnover per instrument
    rx_turnover, rx_avg_yearly_lots, rx_avg_abs_pos = compute_turnover(rx_df)
    ad_turnover, ad_avg_yearly_lots, ad_avg_abs_pos = compute_turnover(ad_df)

    # --- Portfolio (50/50) raw & executed metrics and vols
    port_exec = (WEIGHTS["RX1"] * rx_df["strategy_ret"].fillna(0.0) +
                 WEIGHTS["AD1"] * ad_df["strategy_ret"].fillna(0.0))
    port_raw  = (WEIGHTS["RX1"] * rx_df["forecast_x_return"].fillna(0.0) +
                 WEIGHTS["AD1"] * ad_df["forecast_x_return"].fillna(0.0))

    port_metrics_raw  = performance_metrics(port_raw).add_prefix("raw_")
    port_metrics_exec = performance_metrics(port_exec).add_prefix("exec_")
    port_metrics = pd.concat([port_metrics_raw, port_metrics_exec])

    port_vol_daily_exec = daily_vol(port_exec)
    port_vol_daily_raw  = daily_vol(port_raw)
    port_vol_ann_exec   = _annualize_vol(port_vol_daily_exec)
    port_vol_ann_raw    = _annualize_vol(port_vol_daily_raw)

    # --- Console prints (core stuff)
    print(f"RX1  RAW  Sharpe={rx_metrics['raw_sharpe']:.3f}  Sortino={rx_metrics['raw_sortino']:.3f}  MDD={rx_metrics['raw_max_dd']:.2%}  Vol={rx_vol_ann_raw:.2%}")
    print(f"RX1  EXEC Sharpe={rx_metrics['exec_sharpe']:.3f} Sortino={rx_metrics['exec_sortino']:.3f} MDD={rx_metrics['exec_max_dd']:.2%} Vol={rx_vol_ann_exec:.2%}")
    print(f"     MAE={rx_mae_mfe['mae_mean']:.4f}  MFE={rx_mae_mfe['mfe_mean']:.4f}  Edge={rx_mae_mfe['edge_ratio']:.3f}")
    print(f"     TURNOVER RX1 = {rx_turnover:.2f} | yearly lots={rx_avg_yearly_lots:.1f} | avg|pos|={rx_avg_abs_pos:.1f}")

    print(f"AD1  RAW  Sharpe={ad_metrics['raw_sharpe']:.3f}  Sortino={ad_metrics['raw_sortino']:.3f}  MDD={ad_metrics['raw_max_dd']:.2%}  Vol={ad_vol_ann_raw:.2%}")
    print(f"AD1  EXEC Sharpe={ad_metrics['exec_sharpe']:.3f} Sortino={ad_metrics['exec_sortino']:.3f} MDD={ad_metrics['exec_max_dd']:.2%} Vol={ad_vol_ann_exec:.2%}")
    print(f"     MAE={ad_mae_mfe['mae_mean']:.4f}  MFE={ad_mae_mfe['mfe_mean']:.4f}  Edge={ad_mae_mfe['edge_ratio']:.3f}")
    print(f"     TURNOVER AD1 = {ad_turnover:.2f} | yearly lots={ad_avg_yearly_lots:.1f} | avg|pos|={ad_avg_abs_pos:.1f}")

    print(f"PORT RAW  Sharpe={port_metrics['raw_sharpe']:.3f}  Sortino={port_metrics['raw_sortino']:.3f}  MDD={port_metrics['raw_max_dd']:.2%}  Vol={port_vol_ann_raw:.2%}")
    print(f"PORT EXEC Sharpe={port_metrics['exec_sharpe']:.3f} Sortino={port_metrics['exec_sortino']:.3f} MDD={port_metrics['exec_max_dd']:.2%} Vol={port_vol_ann_exec:.2%}")

    # --- Summary rows
    # RX1 rows
    summary_rows.append({
        "instrument": "RX1", "type": "raw",  "fast": f, "slow": s, "scaler": sc,
        **rx_metrics.filter(like="raw_").rename(lambda k: k.replace("raw_", "")).to_dict(),
        "daily_vol": rx_vol_daily_raw, "ann_vol_check": rx_vol_ann_raw,
        "mae_mean": rx_mae_mfe["mae_mean"], "mfe_mean": rx_mae_mfe["mfe_mean"], "edge_ratio": rx_mae_mfe["edge_ratio"],
        "turnover": rx_turnover, "avg_yearly_lots": rx_avg_yearly_lots, "avg_abs_pos": rx_avg_abs_pos
    })
    summary_rows.append({
        "instrument": "RX1", "type": "exec", "fast": f, "slow": s, "scaler": sc,
        **rx_metrics.filter(like="exec_").rename(lambda k: k.replace("exec_", "")).to_dict(),
        "daily_vol": rx_vol_daily_exec, "ann_vol_check": rx_vol_ann_exec,
        "turnover": rx_turnover, "avg_yearly_lots": rx_avg_yearly_lots, "avg_abs_pos": rx_avg_abs_pos
    })

    # AD1 rows
    summary_rows.append({
        "instrument": "AD1", "type": "raw",  "fast": f, "slow": s, "scaler": sc,
        **ad_metrics.filter(like="raw_").rename(lambda k: k.replace("raw_", "")).to_dict(),
        "daily_vol": ad_vol_daily_raw, "ann_vol_check": ad_vol_ann_raw,
        "mae_mean": ad_mae_mfe["mae_mean"], "mfe_mean": ad_mae_mfe["mfe_mean"], "edge_ratio": ad_mae_mfe["edge_ratio"],
        "turnover": ad_turnover, "avg_yearly_lots": ad_avg_yearly_lots, "avg_abs_pos": ad_avg_abs_pos
    })
    summary_rows.append({
        "instrument": "AD1", "type": "exec", "fast": f, "slow": s, "scaler": sc,
        **ad_metrics.filter(like="exec_").rename(lambda k: k.replace("exec_", "")).to_dict(),
        "daily_vol": ad_vol_daily_exec, "ann_vol_check": ad_vol_ann_exec,
        "turnover": ad_turnover, "avg_yearly_lots": ad_avg_yearly_lots, "avg_abs_pos": ad_avg_abs_pos
    })

    # Portfolio rows (no MAE/MFE defined at portfolio level)
    summary_rows.append({
        "instrument": "PORT", "type": "raw",  "fast": f, "slow": s, "scaler": sc,
        **port_metrics.filter(like="raw_").rename(lambda k: k.replace("raw_", "")).to_dict(),
        "daily_vol": port_vol_daily_raw, "ann_vol_check": port_vol_ann_raw
    })
    summary_rows.append({
        "instrument": "PORT", "type": "exec", "fast": f, "slow": s, "scaler": sc,
        **port_metrics.filter(like="exec_").rename(lambda k: k.replace("exec_", "")).to_dict(),
        "daily_vol": port_vol_daily_exec, "ann_vol_check": port_vol_ann_exec
    })

# =========================================================
# SAVE SUMMARY
# =========================================================
summary_df = pd.DataFrame(summary_rows)
summary_path = os.path.join(OUTPUT_DIR, "gpt_perf_summary_rx1_ad1_with_mae_mfe.csv")
summary_df.to_csv(summary_path, index=False)

print("\n================= SUMMARY (core columns) =================")
cols_to_show = [
    "instrument","type","fast","slow","scaler",
    "sharpe","sortino","max_dd","calmar","ann_return","ann_vol","hit_rate",
    "daily_vol","ann_vol_check","mae_mean","mfe_mean","edge_ratio",
    "turnover","avg_yearly_lots","avg_abs_pos"
]
existing_cols = [c for c in cols_to_show if c in summary_df.columns]
print(summary_df[existing_cols].fillna("").to_string(index=False))

print("\n✅ Saved performance summary to:", summary_path)
print("✅ Per-instrument per-speed CSVs (and trades MAE/MFE tables) saved in:", OUTPUT_DIR)



=== EWMA 2/8 (scaler=10.6) ===
RX1  RAW  Sharpe=0.053  Sortino=0.060  MDD=-63.65%  Vol=48.47%
RX1  EXEC Sharpe=0.010 Sortino=0.011 MDD=-22.26% Vol=10.85%
     MAE=0.4478  MFE=1.0159  Edge=2.269
     TURNOVER RX1 = 58.52 | yearly lots=11627.5 | avg|pos|=99.3
AD1  RAW  Sharpe=-0.791  Sortino=-1.000  MDD=-88.65%  Vol=80.17%
AD1  EXEC Sharpe=-0.984 Sortino=-1.133 MDD=-21.41% Vol=8.55%
     MAE=0.4049  MFE=0.7142  Edge=1.764
     TURNOVER AD1 = 66.34 | yearly lots=9043.3 | avg|pos|=68.2
PORT RAW  Sharpe=-0.649  Sortino=-0.829  MDD=-69.12%  Vol=46.43%
PORT EXEC Sharpe=-0.606 Sortino=-0.683 MDD=-17.75% Vol=6.81%

=== EWMA 4/16 (scaler=7.5) ===
RX1  RAW  Sharpe=-0.567  Sortino=-0.634  MDD=-62.93%  Vol=43.50%
RX1  EXEC Sharpe=-0.345 Sortino=-0.369 MDD=-18.05% Vol=8.37%
     MAE=0.7564  MFE=1.2952  Edge=1.712
     TURNOVER RX1 = 30.50 | yearly lots=5496.7 | avg|pos|=90.1
AD1  RAW  Sharpe=-0.918  Sortino=-1.179  MDD=-90.77%  Vol=75.35%
AD1  EXEC Sharpe=-1.098 Sortino=-1.420 MDD=-18.30% Vol=6.87%